# Calculation of dRMSD reference distances
In this notebook we calculate the distances of IRE1 dimer conformation we took as reference for the dRMSD

In [1]:
import nglview as nv
import mdtraj as md
import numpy as np

# Select the Connections for the DRMSD calculation

## Load structure

In [2]:
# Load the initial structure of the protein
gro = md.load('../../structures/ire1_martini_2monomer_sep_elastic.pdb')

In [3]:
# get the gro file in an array
atoms_all = np.asanyarray(gro.top.to_dataframe()[0])
gro.top.to_dataframe()[0]

,serial,name,element,resSeq,resName,chainID,segmentID
0,1,BB,B,501,PRO,0,PROA
1,2,SC1,S,501,PRO,0,PROA
2,3,BB,B,502,GLU,0,PROA
3,4,SC1,S,502,GLU,0,PROA
4,5,BB,B,503,ASP,0,PROA
...,...,...,...,...,...,...,...
369,370,BB,B,577,LYS,0,PROB
370,371,SC1,S,577,LYS,0,PROB
371,372,SC2,S,577,LYS,0,PROB
372,373,BB,B,578,GLU,0,PROB


## Select connections

In [4]:
# select the atoms to use for the drmsd calculation
drmsd_connections = atoms_all[atoms_all[:,1] == 'BB']
drmsd_connections = drmsd_connections[(drmsd_connections[:,3] <= 558)*
                                      (drmsd_connections[:,3] >= 527)]
monA = drmsd_connections[drmsd_connections[:,-1] == 'PROA']
monB = drmsd_connections[drmsd_connections[:,-1] == 'PROB']
monA_index = monA[:, 0].astype(int)-1
monB_index = monB[:, 0].astype(int)-1

In [6]:
test = np.load('../../DRMSD_reference/CV_connections/connections.npy')

In [9]:
# visualize structure and connections
show_structure = nv.show_mdtraj(gro)
show_structure.add_spacefill(radius_type="vdw", selection=test[:, 0], color='red', opacity=0.3)
show_structure.add_spacefill(radius_type="vdw", selection=test[:, 1], color='blue', opacity=0.3)
show_structure.center()
show_structure

NGLWidget()

## Calculate and save connections

In [6]:
connections = []
for i in monA_index:
    for j in monB_index:
        connections.append([i, j])
connections = np.asanyarray(connections)
print(connections)
np.save('../../DRMSD_reference/CV_connections/connections_cut.npy', connections)

[[ 62 249]
 [ 62 251]
 [ 62 253]
 ...
 [136 318]
 [136 320]
 [136 323]]


## Load reference model

In [7]:
ref = md.load('../../DRMSD_reference/ref.gro')

In [8]:
show_structure = nv.show_mdtraj(ref)
show_structure.remove_backbone()
show_structure.add_representation('cartoon')
show_structure.add_spacefill(radius_type="vdw", selection=monA_index, color='red', opacity=0.3)
show_structure.add_spacefill(radius_type="vdw", selection=monB_index, color='blue', opacity=0.3)
show_structure.center()
show_structure

NGLWidget()

## Calculate and save connections with reference model

In [9]:
connections_ref = []
for i in monA_index:
    for j in monB_index:
        connections_ref.append([i, j, md.compute_distances(ref, [[i,j]])[0][0]])
connections_ref = np.asanyarray(connections_ref)
print(connections_ref)
np.save('../../DRMSD_reference/CV_connections/connections_cut_ref.npy', connections_ref)
print(np.load('../../DRMSD_reference/CV_connections/connections_cut_ref.npy'))

[[ 62.         249.           3.87944579]
 [ 62.         251.           3.76503682]
 [ 62.         253.           3.4793818 ]
 ...
 [136.         318.           1.62209725]
 [136.         320.           1.80449438]
 [136.         323.           2.0610919 ]]
[[ 62.         249.           3.87944579]
 [ 62.         251.           3.76503682]
 [ 62.         253.           3.4793818 ]
 ...
 [136.         318.           1.62209725]
 [136.         320.           1.80449438]
 [136.         323.           2.0610919 ]]
